In [ ]:
import torch

In [ ]:
def get_pretrained_model(model_name: str, num_classes: int, pretrained: bool = True):
    # model_name: 'alexnet', 'vgg11', 'googlenet', 'resnet18'
    
    # 1. Загружаем базовую модель (всегда с дефолтным числом классов ImageNet = 1000)
    model = torch.hub.load('pytorch/vision:v0.10.0', model_name, pretrained=pretrained)
    
    # 2. Адаптируем последний слой под нужное нам количество классов (num_classes)
    if model_name == 'resnet18':
        in_features = model.fc.in_features
        model.fc = torch.nn.Linear(in_features, num_classes)
        
    elif model_name in ['alexnet', 'vgg11']:
        # У AlexNet и VGG последний слой находится внутри блока classifier (индекс 6)
        in_features = model.classifier[6].in_features
        model.classifier[6] = torch.nn.Linear(in_features, num_classes)
        
    elif model_name == 'googlenet':
        in_features = model.fc.in_features
        model.fc = torch.nn.Linear(in_features, num_classes)
        
        # ВАЖНО: У GoogLeNet при обучении могут работать вспомогательные выходы (aux_logits).
        # Если они включены, их классификаторы тоже нужно заменить, иначе будет ошибка.
        if hasattr(model, 'aux1') and model.aux1 is not None:
            model.aux1.fc = torch.nn.Linear(model.aux1.fc.in_features, num_classes)
        if hasattr(model, 'aux2') and model.aux2 is not None:
            model.aux2.fc = torch.nn.Linear(model.aux2.fc.in_features, num_classes)
            
    else:
        raise ValueError(f"Модель {model_name} не поддерживается этой функцией.")
        
    return model